<a href="https://colab.research.google.com/github/doney25/CaseStudy/blob/main/Data_Acquisition_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import pandas as pd
import requests
import datetime
import numpy as np
import os
import time
import sqlite3

In [74]:
def extract():
  population=pd.read_csv('/content/population.csv')
  print("Population Dataset")
  print(population.head(5))
  print(type(population))
  print(population.shape)

  gdp=pd.read_excel('/content/gdp.xlsx')
  print("GDP Dataset")
  print(gdp.head(5))
  print(gdp.columns)
  print(gdp.dtypes)

  internet_users=pd.read_json('/content/internet_users.json')

  literacy_rate = pd.read_xml('/content/literacy_rate.xml')


  return population,gdp,internet_users,literacy_rate


def transform(population, gdp, internet_users, literacy_rate):
  population.rename(columns={"Country": "country"}, inplace=True)
  gdp.rename(columns={"Country": "country"}, inplace=True)
  internet_users.rename(columns={"Country": "country"}, inplace=True)
  literacy_rate.rename(columns={"name": "country"}, inplace=True)

  population["population"] = pd.to_numeric(population["population"])
  gdp["GDP"] = pd.to_numeric(gdp["GDP"])
  internet_users["internet_users"] = pd.to_numeric(internet_users["internet_users"])
  literacy_rate["literacy_rate"] = pd.to_numeric(literacy_rate["literacy_rate"])

  df = population.merge(gdp, on="country")
  df = df.merge(internet_users, on="country")
  df = df.merge(literacy_rate, on="country")

  df["Internet Penetration (%)"] = round((df["internet_users"] / df["population"]) * 100, 2)
  df["GDP Per Capita"] = round(df["GDP"] / df["population"], 2)
  print(f"Transformed {len(df)} rows")

  return df

def load(df):
  conn = sqlite3.connect("country_statistics.db")
  df.to_sql("country_statistics",conn,if_exists="replace",index=False)
  print(f"Loaded {len(df)} rows into SQLite database")
  result = pd.read_sql("SELECT * FROM country_statistics LIMIT 5",conn)
  print(result)
  df.to_csv("final_country_statistics.csv", index=False)
  return conn



In [75]:
population, gdp, internet_users, literacy_rate = extract()

final_df = transform(population, gdp, internet_users, literacy_rate)

conn =load(final_df)

Population Dataset
         country  population
0  United States   331002651
1          China  1411778724
2          India  1380004385
3          Japan   125836021
4        Germany    83240525
<class 'pandas.core.frame.DataFrame'>
(25, 2)
GDP Dataset
         Country       GDP
0  United States  30500000
1          China  19230000
2          India   4187000
3          Japan   4180000
4        Germany   4744000
Index(['Country', 'GDP'], dtype='object')
Country    object
GDP         int64
dtype: object
Transformed 25 rows
Loaded 25 rows into SQLite database
         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   
4        Germany    83240525   4744000        79000000           99.0   



1. List the Top 5 countries by GDP.


In [76]:
print(final_df.sort_values(by="GDP", ascending=False).head(5))

         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
4        Germany    83240525   4744000        79000000           99.0   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   

   Internet Penetration (%)  GDP Per Capita  
0                     94.26            0.09  
1                     76.50            0.01  
4                     94.91            0.06  
2                     61.59            0.00  
3                     93.77            0.03  


2.List the Top 5 countries by population.

In [77]:
print(final_df.sort_values(by="population", ascending=False).head(5))

          country  population       GDP  internet_users  literacy_rate  \
1           China  1411778724  19230000      1080000000           97.2   
2           India  1380004385   4187000       850000000           77.7   
0   United States   331002651  30500000       312000000           99.0   
15      Indonesia   273523615   1500000       212000000           96.0   
10         Brazil   212559417   2330000       181000000           93.2   

    Internet Penetration (%)  GDP Per Capita  
1                      76.50            0.01  
2                      61.59            0.00  
0                      94.26            0.09  
15                     77.51            0.01  
10                     85.15            0.01  


3. Calculate the average literacy rate.

In [78]:
print(final_df["literacy_rate"].mean())

97.248


4. List all countries having a literacy rate greater than 90%.

In [79]:
print(final_df[final_df["literacy_rate"] > 90][["country", "literacy_rate"]])

           country  literacy_rate
0    United States           99.0
1            China           97.2
3            Japan           99.0
4          Germany           99.0
5   United Kingdom           99.0
6           France           99.0
7            Italy           99.0
8           Canada           99.0
9        Australia           99.0
10          Brazil           93.2
11          Russia           99.7
12     South Korea           98.8
13           Spain           98.6
14          Mexico           95.4
15       Indonesia           96.0
16    Saudi Arabia           97.3
17          Turkey           96.7
18    South Africa           95.0
19       Argentina           99.1
20     Netherlands           99.0
21     Switzerland           99.0
22          Sweden           99.0
23          Norway          100.0
24       Singapore           97.5


5. List all countries having an internet penetration greater than 70%.


In [80]:
print(final_df[final_df["Internet Penetration (%)"] > 70][["country", "Internet Penetration (%)"]])

           country  Internet Penetration (%)
0    United States                     94.26
1            China                     76.50
3            Japan                     93.77
4          Germany                     94.91
5   United Kingdom                     97.22
6           France                     94.98
7            Italy                     92.62
8           Canada                     94.72
9        Australia                     93.43
10          Brazil                     85.15
11          Russia                     85.65
12     South Korea                     96.56
13           Spain                     94.11
14          Mexico                     79.11
15       Indonesia                     77.51
16    Saudi Arabia                     97.66
17          Turkey                     81.81
18    South Africa                     72.50
19       Argentina                     90.72
20     Netherlands                     97.47
21     Switzerland                     98.21
22        

6. List the Top 5 countries by GDP per capita.

In [81]:
print(final_df.sort_values(by="GDP Per Capita", ascending=False)[["country", "GDP Per Capita"]].head(5))

          country  GDP Per Capita
21    Switzerland            0.11
23         Norway            0.11
24      Singapore            0.09
0   United States            0.09
9       Australia            0.07


7. Calculate the total population represented in the dataset.


In [82]:
print(final_df["population"].sum())

4720039725


8. Calculate the average internet penetration across all countries.


In [83]:
print(final_df["Internet Penetration (%)"].mean())

89.6596


9. Identify the country having the highest number of internet users.


In [84]:
print(final_df.loc[final_df["internet_users"].idxmax()][["country", "internet_users"]])

country                China
internet_users    1080000000
Name: 1, dtype: object


10. Find the correlation between Literacy Rate and Internet Penetration.

In [85]:
print(final_df["literacy_rate"].corr(final_df["Internet Penetration (%)"]))

0.77279756813985


11. Display the Top 5 countries by GDP.

In [86]:
query = """
SELECT country, gdp
FROM country_statistics
ORDER BY gdp DESC
LIMIT 5;
"""

print(pd.read_sql(query,conn))

         country       GDP
0  United States  30500000
1          China  19230000
2        Germany   4744000
3          India   4187000
4          Japan   4180000


12. Find the average literacy rate.


In [87]:
query = """
SELECT AVG(literacy_rate) AS average_literacy_rate
FROM country_statistics;
"""

print(pd.read_sql(query, conn))

   average_literacy_rate
0                 97.248


13. Display all countries having literacy rate above 90%.

In [88]:
query = """
SELECT country, literacy_rate
FROM country_statistics
WHERE literacy_rate > 90;
"""

print(pd.read_sql(query, conn))

           country  literacy_rate
0    United States           99.0
1            China           97.2
2            Japan           99.0
3          Germany           99.0
4   United Kingdom           99.0
5           France           99.0
6            Italy           99.0
7           Canada           99.0
8        Australia           99.0
9           Brazil           93.2
10          Russia           99.7
11     South Korea           98.8
12           Spain           98.6
13          Mexico           95.4
14       Indonesia           96.0
15    Saudi Arabia           97.3
16          Turkey           96.7
17    South Africa           95.0
18       Argentina           99.1
19     Netherlands           99.0
20     Switzerland           99.0
21          Sweden           99.0
22          Norway          100.0
23       Singapore           97.5


14. Find the Top 5 countries with the highest internet penetration.

In [89]:
query = """
SELECT country, "Internet Penetration (%)"
FROM country_statistics
ORDER BY "Internet Penetration (%)" DESC
LIMIT 5;
"""

print(pd.read_sql(query, conn))

        country  Internet Penetration (%)
0   Switzerland                     98.21
1        Norway                     97.76
2  Saudi Arabia                     97.66
3        Sweden                     97.55
4   Netherlands                     97.47


15.  Calculate the total population.


In [91]:
query = """
SELECT SUM(population) AS total_population
FROM country_statistics;
"""

print(pd.read_sql(query, conn))

   total_population
0        4720039725


16. Find the country with the highest GDP per capita.

In [90]:
query = """
SELECT country, "GDP Per Capita"
FROM country_statistics
ORDER BY "GDP Per Capita" DESC
LIMIT 1;
"""

print(pd.read_sql(query, conn))

       country  GDP Per Capita
0  Switzerland            0.11


17. Export the final merged dataset as a CSV file

In [92]:
query = """
SELECT *
FROM country_statistics;
"""

final_data = pd.read_sql(query, conn)

final_data.to_csv("final_country_statistics.csv", index=False)

print("CSV exported successfully.")

CSV exported successfully.


18. Sort the dataset in descending order of GDP per Capita and display the Top 10
countries.

In [93]:
query = """
SELECT country, "GDP Per Capita"
FROM country_statistics
ORDER BY "GDP Per Capita" DESC
LIMIT 10;
"""

print(pd.read_sql(query, conn))

          country  GDP Per Capita
0     Switzerland            0.11
1          Norway            0.11
2   United States            0.09
3       Singapore            0.09
4       Australia            0.07
5     Netherlands            0.07
6         Germany            0.06
7  United Kingdom            0.06
8          Canada            0.06
9          Sweden            0.06


In [94]:
conn.close()

Extracted data from four different files: CSV (Population), Excel (GDP), JSON (Internet Users), and XML (Literacy Rate).
I cleaned the data by renaming the columns, converting the required columns to numeric data types, and making sure all datasets had a common country column. Merged all four datasets into one DataFrame and created two new columns: Internet Penetration (%) and GDP Per Capita. I loaded the final merged dataset into a SQLite database named country_statistics and checked whether the data was stored correctly. Used both Pandas and SQL queries to analyze the data, such as finding the top countries by GDP and population, average literacy rate, and internet penetration.From the analysis, I observed that countries with higher literacy generally had higher internet penetration, and GDP per capita helped compare the economic performance of different countries.